# 01 — Data inventory (Zenodo corpus, networks, Yₑ controllers)

The acquisition node: the Grichener et al. 2025 reproducibility package
(Zenodo 14873443, 115 GB extracted) plus the two network definitions this
project targets. Also the membership asymmetry that shapes size-transfer and
loss weighting: **mesa_80's weak sector is materially thinner than
mesa_151's**.

Exploratory only — sizes/counts are the RESULTS.md 2026-07-08 rows;
membership below is a lookup against the authoritative isotope lists.

⚠ **Open conflict (reported, not resolved).** This node's status keys to the
phase0-checklist local-confirmation bullet *"Exact mesa_80/mesa_151 isotope
lists … and which Yₑ-controllers each contains"*, which is still unticked —
yet RESULTS.md 2026-07-08 carries both the counts (80/151) and the full
controller membership table reproduced below. So the badge above may read
NOT BUILT while the figures show the measurement. Per docs/CLAUDE.md Rule 0
the docs win until a human decides: either tick the bullet naming those rows
(Rule 3), or state what it still wants. Do not assume the notebook is wrong.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np
import yaml

import nbsupport as nbs

nbs.style()
NETS = ["mesa_80", "mesa_151"]

In [ ]:
nbs.provenance_header(
    "01",
    "Data inventory — Zenodo corpus, networks, Yₑ controllers",
    nbs.status_of([], ["Exact mesa_80/mesa_151 isotope lists"]),
    results_rows=[
        "2026-07-08: zip 49.06 GB, md5 ab31e569… (manifest match); 115 148 407 532 B extracted, 3605 files",
        "2026-07-08: second-level breakdown — training_sets 83.73 GB, trained_NNN_models 23.80 GB, test_datasets 6.07 GB, python_scripts 0.99 GB, MESA_models 0.55 GB",
        "2026-07-08: exact isotope counts 80 / 151 (ca41 present in BOTH; paper App. A omits it — paper typo)",
        "2026-07-08: Yₑ-controller membership — mesa_80 carries 6/9 EC controllers, 0/8 β-decay partners",
    ],
    data=[
        "data/MANIFEST.yaml (Zenodo 14873443)",
        "configs/isotopes_mesa{80,151}.yaml (sourced from test-set column headers)",
    ],
    scripts=["scripts/download_zenodo.py", "scripts/check_training_csvs.py"],
)

## Figure 1 — what the 115 GB actually is

Directory sizes measured on disk; they reproduce the RESULTS.md
second-level breakdown row. Note the shape of the problem: the training sets
dominate, and they are **non-regenerable** (unseeded-scrambled Sobol — the
shipped file is the only ground truth; joins key on `state_id` only).

In [ ]:
ZEN = nbs.REPO / "data" / "zenodo" / "NuclearNeuralNetworks"


def dir_bytes(p: Path) -> int:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


parts = {p.name: dir_bytes(p) for p in sorted(ZEN.iterdir()) if p.is_dir()}
parts = dict(sorted(parts.items(), key=lambda kv: kv[1]))

fig, ax = plt.subplots(figsize=(9, 3.6))
y = np.arange(len(parts))
ax.barh(y, [v / 1e9 for v in parts.values()], color="#0072B2")
ax.set_yticks(y, list(parts))
for i, v in enumerate(parts.values()):
    ax.text(v / 1e9 + 1, i, f"{v / 1e9:.2f} GB", va="center", fontsize=8)
ax.set_xlabel("GB on disk")
ax.set_xlim(0, 95)
ax.set_title(f"Zenodo 14873443 extracted — {sum(parts.values()) / 1e9:.1f} GB total")
nbs.caption(
    fig,
    "The 83.7 GB of training_sets are the 2²⁰ states/net × 9 dt Sobol grid — NON-REGENERABLE "
    "(unseeded-scrambled Sobol; the shipped file is the only ground truth, and anything keyed to "
    "states must persist explicit state_id lists). test_datasets holds the trajectory files used "
    "in notebooks 07–09; trained_NNN_models is the baseline reproduced in notebook 02.",
    results=["RESULTS.md 2026-07-08 extraction + breakdown rows"],
    scripts=["scripts/download_zenodo.py"],
)

## Figure 2 — the two networks on the (N, Z) plane

mesa_151 extends mesa_80 mostly into the **neutron-rich Fe-peak**, which is
exactly where the Yₑ-controlling weak reactions live. `sc45` — the high-Yₑ
bottleneck reactant ⁴⁵Sc(p,γ)⁴⁶Ti — is absent from mesa_80 (whose only Sc is
⁴³Sc), so that kill-test instrument is mesa_151-only.

In [ ]:
iso = {}
for net in NETS:
    d = yaml.safe_load((nbs.REPO / "configs" / f"isotopes_{net.replace('_', '')}.yaml").read_text())
    iso[net] = {e["name"]: (e["Z"], e["A"]) for e in d["isotopes"]}
    print(f"{net}: {d['n_isotopes']} isotopes  (yaml n_isotopes field: {d['n_isotopes']})")

only151 = set(iso["mesa_151"]) - set(iso["mesa_80"])
print(f"mesa_151-only species: {len(only151)}")
print("ca41 in both:", "ca41" in iso["mesa_80"], "ca41" in iso["mesa_151"])

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for net, marker, size, color, label in [
    ("mesa_151", "s", 46, "#E69F00", "mesa_151 only"),
    ("mesa_80", "s", 46, "#0072B2", "mesa_80 (⊂ mesa_151)"),
]:
    Z = np.array([v[0] for v in iso[net].values()])
    A = np.array([v[1] for v in iso[net].values()])
    ax.scatter(A - Z, Z, marker=marker, s=size, c=color, label=label, edgecolors="none")
for name, c in [("sc45", "#D55E00"), ("ca41", "#009E73")]:
    if name in iso["mesa_151"]:
        z, a = iso["mesa_151"][name]
        ax.scatter([a - z], [z], s=150, facecolors="none", edgecolors=c, linewidths=2)
        note = "sc45 — mesa_151 ONLY\n(⁴⁵Sc(p,γ)⁴⁶Ti bottleneck)" if name == "sc45" else (
            "ca41 — in BOTH nets\n(paper App. A typo)")
        ax.annotate(note, (a - z, z), textcoords="offset points", xytext=(12, -22),
                    fontsize=8, color=c,
                    arrowprops=dict(arrowstyle="-", color=c, lw=1))
ax.plot([0, 40], [0, 40], ls=":", c="0.6", lw=1)
ax.text(33, 34, "N = Z", fontsize=8, color="0.5")
ax.set_xlabel("neutron number N = A − Z")
ax.set_ylabel("proton number Z")
ax.set_title("Network coverage on the (N, Z) plane")
ax.legend(loc="upper left")
nbs.caption(
    fig,
    "mesa_80 (80 species) is a strict subset of mesa_151 (151); the extra 71 species are "
    "predominantly neutron-rich Fe-peak nuclei — the sector that carries Yₑ evolution. This "
    "asymmetry is why zero-shot mesa_80→151 transfer is a HYPOTHESIS with a falsifier "
    "(checklist row 5: >2× internal Yₑ error ⇒ report size-ADAPTABLE, not size-TRANSFERABLE).",
    results=[
        "RESULTS.md 2026-07-08 isotope-count row (80/151; ca41 in both — paper typo)",
        "RESULTS.md 2026-07-08 Yₑ-controller membership table (sc45 absent from mesa_80)",
    ],
    scripts=["configs/isotopes_mesa{80,151}.yaml (headers are authoritative)"],
)

## Figure 3 — Yₑ-controller membership: mesa_80's thin weak sector

The controller list below is **sourced** from the RESULTS.md 2026-07-08
membership table (Step-2 task spec: EC controllers + β-decay partners);
membership itself is recomputed here as a lookup against the isotope yamls,
so this figure cannot drift from the network definitions.

In [ ]:
# sourced: RESULTS.md 2026-07-08 Yₑ-controller membership table (roles per Step-2 spec)
CONTROLLERS = [
    ("co55", "EC"), ("ni56", "EC"), ("fe55", "EC"), ("fe54", "EC"), ("v51", "EC"),
    ("cr53", "EC"), ("s33", "EC"), ("cl35", "EC"), ("ar37", "EC"),
    ("mn56", "β partner"), ("cr56", "β partner"), ("fe59", "β partner"),
    ("fe61", "β partner"), ("co61", "β partner"), ("co63", "β partner"),
    ("co60", "β partner"), ("co59", "β partner"),
]

M = np.array([[1.0 if name in iso[net] else 0.0 for net in NETS] for name, _ in CONTROLLERS])

fig, ax = plt.subplots(figsize=(6.4, 6.2))
ax.imshow(M, cmap=plt.cm.colors.ListedColormap(["#EEEEEE", "#009E73"]), aspect="auto",
          vmin=0, vmax=1)
ax.set_xticks(range(len(NETS)), NETS)
ax.set_yticks(range(len(CONTROLLERS)), [f"{n}  ({r})" for n, r in CONTROLLERS], fontsize=8.5)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, "yes" if M[i, j] else "NO", ha="center", va="center",
                fontsize=8, color="white" if M[i, j] else "#B3261E",
                fontweight="normal" if M[i, j] else "bold")
ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(CONTROLLERS), 1), minor=True)
ax.grid(which="minor", color="white", lw=1.5)
ax.grid(which="major", visible=False)
ax.set_title("Yₑ-controller membership by network")

ec = [i for i, (_, r) in enumerate(CONTROLLERS) if r == "EC"]
bp = [i for i, (_, r) in enumerate(CONTROLLERS) if r != "EC"]
counts = {net: (int(M[ec, j].sum()), int(M[bp, j].sum())) for j, net in enumerate(NETS)}
print("EC controllers / β partners present:",
      {k: f"{v[0]}/{len(ec)} EC, {v[1]}/{len(bp)} β" for k, v in counts.items()})
nbs.caption(
    fig,
    f"mesa_80 carries {counts['mesa_80'][0]}/{len(ec)} EC controllers and "
    f"{counts['mesa_80'][1]}/{len(bp)} β-decay partners; mesa_151 carries "
    f"{counts['mesa_151'][0]}/{len(ec)} and {counts['mesa_151'][1]}/{len(bp)}. cr56 and co63 are in "
    "NEITHER shipped network. mesa_80's Yₑ evolution therefore runs through a materially thinner "
    "weak set — feeds size-transfer design and loss weighting (⁵⁶Ni EC is the #1 |dẎₑ| channel, "
    "notebook 10).",
    results=["RESULTS.md 2026-07-08 Yₑ-controller membership table"],
    scripts=["(lookup against configs/isotopes_mesa{80,151}.yaml)"],
)

## TODO (stub)

- **Training-CSV column census** — per-dt column schema, eps normalization
  (÷1e16 in the CSVs, unlike the trajectory files: notebook 07), and the
  `state_id` join discipline. Producer exists: `scripts/check_training_csvs.py`
  (RESULTS.md 2026-07-08 training-CSV audit rows).

## What this notebook does NOT show

- Rate-level weak inventory (which tables, LMP > Oda > FFN): notebook 04.
- The Sobol grid's coverage/missing-row structure:
  `scripts/sobol_missing_rows.py`, `docs/figures/sobol_missing_*.png`.